# Trích xuất feature EEG — AF3/AF4 (band power, IAF, Hjorth, entropy, cross-channel)

Input: epoch 2s đã xử lý (`notch + bandpass 1-45Hz + segment rejection`) từ
`notebooks/eeg_qc.ipynb` — `data/meditation_ingestion_output/analytics/eeg_qc/epochs_processed/`.

Mỗi epoch được gán nhãn phase (`REST_BEFORE` / `MEDITATION` / `REST_AFTER`)
theo annotation gắn sẵn từ `event_labels.py`. Feature tính **per-epoch**, sau
đó **aggregate (median) theo (session, phase)** để so sánh trực tiếp với dữ
liệu khảo sát pre/post (notebook khảo sát riêng, không kèm trong repo này).

Nhóm feature (xem thảo luận đầy đủ + trích dẫn nghiên cứu trong hội thoại đã
dẫn tới notebook này):

| Nhóm | Feature | Ghi chú |
|---|---|---|
| Band power | delta/theta/alpha/beta/gamma/**high_beta (20-30Hz)**, **tuyệt đối + tương đối** | tương đối ưu tiên hơn cho wearable (trở kháng đổi theo phiên); high_beta = beta con, liên hệ căng thẳng/hyperarousal (QEEG) |
| Alpha cá nhân hoá | IAF (tần số/công suất/prominence đỉnh alpha) | Klimesch 1999 |
| Tỉ lệ | theta/alpha ratio, **theta/(alpha+beta) ratio** | Gruzelier 2009 (neurofeedback); theta/(alpha+beta): họ chỉ số mệt mỏi tinh thần/buồn ngủ (Jap et al. 2009) |
| Hjorth | Activity, Mobility, Complexity | Hjorth 1970, bền với nhiễu wearable |
| Entropy/phức tạp | DWT (log energy, Shannon, spectral, wavelet entropy, Higuchi FD) × 6 subband, **Rényi entropy (bậc 2, trên PSD)** | `lib/features.py::extract_dwt_features`, db4 level 5; Rényi: tổng quát hoá Shannon, ổn định hơn với mẫu ngắn |
| Cross-channel (AF3-AF4) | FAA, FEI, coherence alpha/beta, **imaginary coherency**, PLV alpha/beta | FAA: Davidson 1992/2003; FEI: Pope 1995 (không chuyên biệt thiền); imaginary coherency: Nolte 2004 (bền với volume conduction) |

`COMPUTE_DWT_FEATURES = True` mặc định (nhóm tốn thời gian tính nhất) — đổi
thành `False` nếu chỉ cần chạy nhanh nhóm band power/Hjorth/cross-channel.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from scipy import signal

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

REPO_ROOT = Path("..").resolve()
LIB_DIR = REPO_ROOT / "lib"
HAPPE_SRC = REPO_ROOT / "happe-python" / "src"
for p in (LIB_DIR, HAPPE_SRC):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import mne
mne.set_log_level("ERROR")

from features import bandpower, compute_plv, extract_dwt_features, _alpha_peak_features_psd

COMPUTE_DWT_FEATURES = True

EEG_QC_DIR = REPO_ROOT / "data" / "analytics" / "eeg_qc"
EPOCHS_DIR = EEG_QC_DIR / "epochs_processed"
OUT_DIR = REPO_ROOT / "data" / "analytics" / "eeg_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CH_NAMES = ["AF3_RAW", "AF4_RAW"]
BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 45),
        "high_beta": (20, 30)}  # high beta/beta3 -- QEEG convention (vd. beta thap 13-20, beta cao 20-30),
                                # lien he voi cang thang/hyperarousal trong mot so nghien cuu neurofeedback.
TOTAL_BAND = (1, 45)

epoch_files = sorted(EPOCHS_DIR.glob("*-epo.fif"))
print(f"Tìm thấy {len(epoch_files)} file epoch")

## 1. Hàm tính feature

In [ ]:
def all_band_powers(x, fs, nperseg=None):
    '''1 lần welch -> công suất tuyệt đối + tương đối mọi băng, để tránh gọi welch lặp lại.'''
    nperseg = nperseg or min(int(fs * 2), len(x))
    freqs, psd = signal.welch(x, fs=fs, nperseg=nperseg, detrend="constant")
    total_idx = (freqs >= TOTAL_BAND[0]) & (freqs <= TOTAL_BAND[1])
    total_power = np.trapz(psd[total_idx], freqs[total_idx])
    out = {}
    for name, (lo, hi) in BANDS.items():
        idx = (freqs >= lo) & (freqs <= hi)
        p = np.trapz(psd[idx], freqs[idx])
        out[f"{name}_power_abs"] = float(p)
        out[f"{name}_power_rel"] = float(p / total_power) if total_power > 0 else np.nan
    out["theta_alpha_ratio"] = (
        out["theta_power_abs"] / out["alpha_power_abs"] if out["alpha_power_abs"] > 0 else np.nan
    )
    # theta/(alpha+beta) -- bien the ho chi so theta/beta dung trong nghien cuu
    # met moi tinh than/buon ngu (vd. Jap et al., 2009, Expert Syst. Appl.).
    denom = out["alpha_power_abs"] + out["beta_power_abs"]
    out["theta_alpha_beta_ratio"] = out["theta_power_abs"] / denom if denom > 0 else np.nan
    return out


def renyi_entropy(x, fs, alpha=2, nperseg=None):
    '''Renyi entropy bac alpha tren PSD chuan hoa thanh phan phoi xac suat
    (alpha=2: "collision/quadratic entropy", pho bien trong EEG vi de uoc luong
    va on dinh hon Shannon voi mau ngan). alpha=1 tro ve gioi han Shannon.'''
    nperseg = nperseg or min(int(fs * 2), len(x))
    freqs, psd = signal.welch(x, fs=fs, nperseg=nperseg, detrend="constant")
    p = psd / (psd.sum() + 1e-12)
    p = p[p > 0]
    if p.size == 0:
        return np.nan
    if abs(alpha - 1.0) < 1e-9:
        return float(-np.sum(p * np.log(p)))
    return float(1.0 / (1.0 - alpha) * np.log(np.sum(p ** alpha) + 1e-12))


def hjorth_params(x):
    '''Hjorth Activity/Mobility/Complexity (Hjorth, 1970).'''
    x = np.asarray(x, dtype=float)
    dx = np.diff(x)
    ddx = np.diff(dx)
    var_x, var_dx, var_ddx = np.var(x), np.var(dx), np.var(ddx)
    activity = var_x
    mobility = np.sqrt(var_dx / var_x) if var_x > 0 else 0.0
    mobility_dx = np.sqrt(var_ddx / var_dx) if var_dx > 0 else 0.0
    complexity = (mobility_dx / mobility) if mobility > 0 else 0.0
    return {"hjorth_activity": float(activity), "hjorth_mobility": float(mobility),
            "hjorth_complexity": float(complexity)}


def imaginary_coherency(x, y, fs, band, nperseg=None):
    '''|Im(coherency)| trung bình trong band -- bền với volume conduction hơn
    coherence thường (Nolte et al., 2004, Clin. Neurophysiol.).'''
    nperseg = nperseg or min(256, len(x))
    f, Pxy = signal.csd(x, y, fs=fs, nperseg=nperseg)
    _, Pxx = signal.welch(x, fs=fs, nperseg=nperseg)
    _, Pyy = signal.welch(y, fs=fs, nperseg=nperseg)
    denom = np.sqrt(Pxx * Pyy)
    with np.errstate(divide="ignore", invalid="ignore"):
        coh_complex = np.where(denom > 0, Pxy / denom, 0.0)
    idx = (f >= band[0]) & (f <= band[1])
    return float(np.mean(np.abs(np.imag(coh_complex[idx])))) if np.any(idx) else np.nan


def bandpass_filter(x, fs, band, order=4):
    nyq = fs / 2.0
    sos = signal.butter(order, [band[0] / nyq, band[1] / nyq], btype="band", output="sos")
    return signal.sosfiltfilt(sos, x)


def cross_channel_features(x, y, fs):
    '''FAA, FEI, coherence, imaginary coherency, PLV -- x=AF3 (trái), y=AF4 (phải).'''
    feats = {}
    bp_x = all_band_powers(x, fs)
    bp_y = all_band_powers(y, fs)

    alpha_x, alpha_y = bp_x["alpha_power_abs"], bp_y["alpha_power_abs"]
    beta_x, beta_y = bp_x["beta_power_abs"], bp_y["beta_power_abs"]
    theta_x, theta_y = bp_x["theta_power_abs"], bp_y["theta_power_abs"]

    # FAA (Davidson): ln(phải) - ln(trái); AF3=trái, AF4=phải (hệ 10-10).
    feats["FAA"] = float(np.log(alpha_y + 1e-8) - np.log(alpha_x + 1e-8))
    # FEI / Engagement Index (Pope et al., 1995) -- KHÔNG chuyên biệt thiền, dùng exploratory.
    feats["FEI"] = float((beta_x + beta_y) / (alpha_x + alpha_y + theta_x + theta_y + 1e-8))

    nperseg = min(256, len(x))
    f, Cxy = signal.coherence(x, y, fs=fs, nperseg=nperseg)
    feats["coh_alpha"] = float(np.mean(Cxy[(f >= 8) & (f <= 12)]))
    feats["coh_beta"] = float(np.mean(Cxy[(f >= 13) & (f <= 30)]))

    feats["icoh_alpha"] = imaginary_coherency(x, y, fs, (8, 12), nperseg)
    feats["icoh_beta"] = imaginary_coherency(x, y, fs, (13, 30), nperseg)

    x_alpha, y_alpha = bandpass_filter(x, fs, (8, 12)), bandpass_filter(y, fs, (8, 12))
    x_beta, y_beta = bandpass_filter(x, fs, (13, 30)), bandpass_filter(y, fs, (13, 30))
    feats["plv_alpha"] = float(compute_plv(x_alpha, y_alpha))
    feats["plv_beta"] = float(compute_plv(x_beta, y_beta))

    return feats


def dwt_feature_dict(x, fs, prefix, wavelet="db4", level=5):
    vals = extract_dwt_features(x, fs=fs, wavelet=wavelet, level=level)
    names = ["log_energy", "shannon_entropy", "spectral_entropy", "wavelet_entropy", "higuchi_fd"]
    n_sub = len(vals) // len(names)
    out = {}
    for s in range(n_sub):
        for j, nm in enumerate(names):
            out[f"{prefix}_dwt_sub{s}_{nm}"] = float(vals[s * len(names) + j])
    return out

## 2. Gán phase cho từng epoch (theo annotation REST_BEFORE/MEDITATION/REST_AFTER)

In [ ]:
def epoch_phase_labels(epochs):
    '''1 nhãn phase / epoch -- lấy annotation có duration lớn nhất đè lên epoch đó
    (loại các point-event duration=0 như SESSION_START/PRECHECK_SUBMITTED...).'''
    per_epoch = epochs.get_annotations_per_epoch()
    labels = []
    for anns in per_epoch:
        candidates = [(dur, desc) for (_onset, dur, desc) in anns if dur > 0]
        if not candidates:
            labels.append("UNLABELED")
            continue
        candidates.sort(key=lambda t: t[0], reverse=True)
        labels.append(candidates[0][1])
    return labels

## 3. Chạy trích feature trên toàn bộ session (per-epoch)

In [ ]:
t0 = time.time()
all_rows = []
n_epochs_total = 0

for fi, f in enumerate(epoch_files):
    stem = f.stem.replace("-epo", "")
    epochs = mne.read_epochs(f, preload=True, verbose="ERROR")
    fs = epochs.info["sfreq"]
    phases = epoch_phase_labels(epochs)
    data = epochs.get_data(picks=CH_NAMES)  # (n_epochs, 2, n_times)

    for ei in range(len(epochs)):
        x, y = data[ei, 0], data[ei, 1]
        row = {"File": f"{stem}.fif", "epoch_idx": ei, "phase": phases[ei]}

        for ch_name, sig in zip(["AF3", "AF4"], [x, y]):
            bp = all_band_powers(sig, fs)
            for k, v in bp.items():
                row[f"{ch_name}_{k}"] = v
            iaf_freq, iaf_power, iaf_prom = _alpha_peak_features_psd(sig, fs=fs)
            row[f"{ch_name}_iaf_freq"] = iaf_freq
            row[f"{ch_name}_iaf_power"] = iaf_power
            row[f"{ch_name}_iaf_prominence"] = iaf_prom
            hj = hjorth_params(sig)
            for k, v in hj.items():
                row[f"{ch_name}_{k}"] = v
            row[f"{ch_name}_renyi_entropy"] = renyi_entropy(sig, fs, alpha=2)
            if COMPUTE_DWT_FEATURES:
                row.update(dwt_feature_dict(sig, fs, ch_name))

        row.update({f"cross_{k}": v for k, v in cross_channel_features(x, y, fs).items()})
        all_rows.append(row)

    n_epochs_total += len(epochs)
    if (fi + 1) % 5 == 0 or fi == len(epoch_files) - 1:
        elapsed = time.time() - t0
        print(f"[{fi+1}/{len(epoch_files)}] {stem} | {len(epochs)} epoch | "
              f"tổng {n_epochs_total} epoch | {elapsed:.0f}s trôi qua")

epoch_features = pd.DataFrame(all_rows)
print(f"\nXong: {len(epoch_features)} epoch x {epoch_features.shape[1]} cột, {time.time()-t0:.0f}s")
epoch_features.head()

## 4. Ghép định danh + lưu bảng feature per-epoch

In [ ]:
import re
PARTICIPANT_RE = re.compile(r"^(.+?)_day")
DAY_RE = re.compile(r"_day(\d+)_")
SESSION_RE = re.compile(r"_day\d+_([0-9a-f]+)\.fif$")

def parse_identity(file_name):
    p = PARTICIPANT_RE.search(file_name)
    d = DAY_RE.search(file_name)
    s = SESSION_RE.search(file_name)
    return pd.Series({
        "participant_key": p.group(1) if p else None,
        "study_day": int(d.group(1)) if d else None,
        "session_id_short": s.group(1) if s else None,
    })

identity = epoch_features["File"].apply(parse_identity)
epoch_features = pd.concat([identity, epoch_features], axis=1)
cols_front = ["participant_key", "study_day", "session_id_short", "File", "epoch_idx", "phase"]
epoch_features = epoch_features[cols_front + [c for c in epoch_features.columns if c not in cols_front]]

epoch_out = OUT_DIR / "features_per_epoch.csv"
epoch_features.to_csv(epoch_out, index=False)
print("Đã lưu:", epoch_out, "-", epoch_features.shape)
print("\nSố epoch theo phase:")
print(epoch_features["phase"].value_counts())

## 5. Aggregate theo (session, phase)

Đơn vị phân tích cho notebook EDA/thống kê tiếp theo: 1 dòng = 1 session ×
1 phase (median các feature qua toàn bộ epoch của phase đó trong session).

In [ ]:
feature_cols = [c for c in epoch_features.columns
               if c not in ("participant_key", "study_day", "session_id_short", "File", "epoch_idx", "phase")]

agg = (
    epoch_features[epoch_features["phase"] != "UNLABELED"]
    .groupby(["participant_key", "study_day", "session_id_short", "File", "phase"], dropna=False)[feature_cols]
    .median()
    .reset_index()
)
n_epochs_per_phase = (
    epoch_features[epoch_features["phase"] != "UNLABELED"]
    .groupby(["participant_key", "study_day", "session_id_short", "File", "phase"], dropna=False)
    .size().reset_index(name="n_epochs")
)
agg = agg.merge(n_epochs_per_phase, on=["participant_key", "study_day", "session_id_short", "File", "phase"])

agg_out = OUT_DIR / "features_per_session_phase.csv"
agg.to_csv(agg_out, index=False)
print("Đã lưu:", agg_out, "-", agg.shape)

coverage = agg.pivot_table(index="File", columns="phase", values="n_epochs", aggfunc="first")
n_full = (coverage.notna().sum(axis=1) == 3).sum() if coverage.shape[1] >= 3 else 0
print(f"\nSession có đủ cả 3 phase (REST_BEFORE/MEDITATION/REST_AFTER): {n_full}/{coverage.shape[0]}")
agg.head()

## 6. Tổng kết & đường dẫn output

- `data/meditation_ingestion_output/analytics/eeg_features/features_per_epoch.csv` — feature từng epoch (đơn vị nhỏ nhất, dùng nếu cần phân tích trong-phase theo thời gian)
- `data/meditation_ingestion_output/analytics/eeg_features/features_per_session_phase.csv` — **bảng chính cho EDA/thống kê tiếp theo** (median feature theo session × phase + `n_epochs`)

Notebook tiếp theo (`eeg_feature_eda_stats.ipynb`) dùng file thứ 2 để: so
sánh REST_BEFORE vs MEDITATION vs REST_AFTER (Wilcoxon, khớp phương pháp đã
dùng cho khảo sát), và đối chiếu với notebook khảo sát pre/post (không kèm trong repo này).

In [ ]:
print("Output directory:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)